In [ ]:
# -*- coding: utf-8 -*-
# ============================================================================
# CELLA 0 - IMPORTS & SETUP (commit epoca 2)
# Batch session (Save & Run All / commit): niente idle timeout, 9h.
# Gestisce il dataset input sia come cartelle sia come ZIP (dir-mode zip).
# ============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Force single-GPU su Kaggle
import sys
import json
import time
import gc
import shutil
import zipfile

# Il dataset input puo' montare in piu' path: provali tutti
_CANDIDATES = [
    "/kaggle/input/andrlab/gemma-4-omni-module",
    "/kaggle/input/datasets/andrlab/gemma-4-omni-module",
    "/kaggle/input/andrlab/gemma4-ep2-data",
    "/kaggle/input/datasets/andrlab/gemma4-ep2-data",
]
_INPUT = None
for c in _CANDIDATES:
    if os.path.isdir(c):
        _INPUT = c
        break
print("Input dataset files:", os.listdir(_INPUT) if _INPUT else "MANCANTE")
print("_INPUT =", _INPUT)
sys.stdout.flush()

def _extract_zip(zpath, dst):
    with zipfile.ZipFile(zpath) as z:
        # estrai mantenendo le sottocartelle come nel zip
        z.extractall(dst)

def _copy_tree(src, dst):
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.islink(s):
            continue
        if os.path.isdir(s):
            os.makedirs(d, exist_ok=True)
            _copy_tree(s, d)
        else:
            os.makedirs(os.path.dirname(d), exist_ok=True)
            if not os.path.exists(d):
                shutil.copy2(s, d)

if _INPUT:
    # moduli python: copia diretta
    for f in ("kaggle_pipeline.py", "code2wav_decoder.py", "extract_qwen_subtalker.py",
              "qwen_subtalker.py", "_epoca2.py"):
        s = os.path.join(_INPUT, f)
        if os.path.isfile(s):
            d = os.path.join("/kaggle/working", f)
            if not os.path.exists(d):
                shutil.copy2(s, d)
    # dati: cartelle o zip
    for sub in ("data", "checkpoints", "code2wav_weights", "qwen_subtalker"):
        s = os.path.join(_INPUT, sub)
        z = os.path.join(_INPUT, sub + ".zip")
        if os.path.isdir(s):
            _copy_tree(s, sub)
        elif os.path.isfile(z):
            os.makedirs(sub, exist_ok=True)
            _extract_zip(z, sub)
            print(f"zip estratto: {sub}.zip -> {sub}/")
    # eventuale file_cleanup
    for f in os.listdir(_INPUT):
        if f.endswith(('.dat', '.log')):
            s = os.path.join(_INPUT, f)
            if not os.path.exists(os.path.join("/kaggle/working", f)):
                shutil.copy2(s, "/kaggle/working")

sys.path.insert(0, "/kaggle/working")
# verifica parquet
import glob
par = glob.glob("data/train_tokens_50.0h_2500s_mel.parquet")
print("parquet 50h presente:", bool(par), par[:2])
sys.stdout.flush()


In [ ]:
# CELLA INSTALL - dipendenze Fase 2 su GPU P100 (sm_60)
# P100 non supporta torch cu128 (solo sm_70+). Serve torch cu118.
# Pattern VERIFICATO in test kernel: torch cu118 + --no-deps per non far toccare torch.
import subprocess, sys, os, re as _re

def gpu_info():
    try:
        r = subprocess.run([sys.executable, '-c',
            "import torch; print(torch.cuda.get_device_name(0)); print(torch.cuda.get_device_capability(0))"],
            capture_output=True, text=True, timeout=120)
        lines = [l for l in r.stdout.strip().splitlines() if l.strip()]
        cap = None
        if len(lines) > 1:
            m = _re.search(r'(\d+)', lines[1])
            cap = int(m.group(1)) if m else None
        return (lines[0] if lines else "?"), cap
    except Exception as e:
        return f"ERR {str(e)[:50]}", None

name, cap = gpu_info()
print("GPU rilevata:", name, "| compute capability:", cap)
sys.stdout.flush()

if cap is not None and cap < 7:
    # P100 (sm_60): stack cu118 (supporta Pascal), torch 2.6.0
    print(">>> GPU Pascal rilevata: installo torch 2.6.0+cu118 (supporta sm_60) <<<")
    sys.stdout.flush()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--index-url', 'https://download.pytorch.org/whl/cu118',
         'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0'],
        capture_output=True, text=True, timeout=1800)
    print("torch cu118 rc:", r.returncode, (r.stdout + r.stderr)[-150:])
    sys.stdout.flush()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
         'transformers==5.5.0', 'unsloth', 'unsloth_zoo', 'peft', 'safetensors', 'soundfile', 'pyarrow',
         'trl', 'xformers==0.0.29.post3', 'bitsandbytes', 'triton', 'tyro', 'structlog',
         'nest-asyncio', 'hf_transfer', 'diffusers', 'typer', 'sentencepiece', 'datasets'],
        capture_output=True, text=True, timeout=1800)
    print("stack cu118 rc:", r.returncode, (r.stdout + r.stderr)[-200:])
else:
    # T4 o superiore (cap>=7): stack normale cu128 (il kernel batch preinstalla torch cu128)
    # NOTA: qui NON usiamo --no-deps perche' le deps di transformers/unsloth (accelerate,
    # tokenizers, huggingface-hub, etc.) devono risolversi. torch resta quello preinstallato
    # (2.10+cu128) perche' transformers 5.5.0 richiede torch>=2.4 e unsloth torch<2.12.
    print(">>> GPU Volta+ rilevata: stack normale cu128 <<<")
    sys.stdout.flush()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         'transformers==5.5.0', 'unsloth', 'safetensors', 'soundfile', 'pyarrow'],
        capture_output=True, text=True, timeout=1800)
    print("stack cu128 rc:", r.returncode, (r.stdout + r.stderr)[-250:])
    sys.stdout.flush()

import importlib.metadata as md
for pkg in ['transformers', 'unsloth', 'peft', 'torch']:
    try: print(pkg + ": " + md.version(pkg))
    except Exception: print(pkg + ": MANCANTE")
# verifica GPU compatibile con torch
r2 = subprocess.run([sys.executable, '-c',
    "import torch; print('cuda ok:', torch.cuda.is_available(), torch.cuda.get_device_name(0))"],
    capture_output=True, text=True, timeout=120)
print("check cuda:", (r2.stdout + r2.stderr).strip()[-200:])


In [ ]:
# -*- coding: utf-8 -*-
# ============================================================================
# CELLA 2 - EPOCA 2 (lr 5e-5 + gradient clipping) da checkpoint epoca 1
# Importa kaggle_pipeline dal working (aggiornato: PHASE2_LR, PHASE2_MAX_GRAD_NORM,
# resume, metrica sintesi generated-vs-oracle). Gira come subprocess nohup robusto.
# ============================================================================
import os, sys, time, subprocess

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
sys.path.insert(0, "/kaggle/working")

import unsloth  # noqa: F401  (patch in processo fresco)

import kaggle_pipeline as KP

print("=== COMMIT EPOCA 2: lr=5e-5, max_grad_norm=1.0, resume da step 12000 ===")
sys.stdout.flush()

# env per il training (visti da kaggle_pipeline.run_training_pipeline)
os.environ["PHASE2_RESUME_CKPT"] = "checkpoints/gemma4_duplex_sota_ckpt"
os.environ["PHASE2_START_STEP"] = "12000"
os.environ["PHASE2_LR"] = "5e-5"
os.environ["PHASE2_MAX_GRAD_NORM"] = "1.0"
os.environ["PHASE2_RUN_LOG"] = "data/phase2_run_log_ep2.tsv"
os.environ["PHASE2_CKPT_LOG"] = "data/phase2_ckpt_log_50h_ep2.txt"

duplex_model, _ = KP.load_gemma4_duplex_system()
KP.run_training_pipeline(duplex_model, num_epochs=1)

print("=== EPOCA 2 COMPLETATA ===")
